In [ ]:
import pandas as pd
import sys
import os
from datetime import datetime
from dotenv import load_dotenv

# Garante que o Python encontre a pasta 'utils' na raiz do projeto
sys.path.append(os.path.abspath(os.path.join('..')))
from utils.minio_client import MinioClient

função de tratamento - artistas mais ouvidos

In [2]:
def transform_artists(data) -> pd.DataFrame:
    """
    Transforma os dados brutos de artistas da Bronze.
    Retorna um DataFrame com os campos disponíveis pelo endpoint atual da API.
    """
    items = data["items"] if isinstance(data, dict) and "items" in data else data

    # achata tudo, criando colunas, pois, o JSON da API do Spotify tem campos dentro de campos
    df = pd.json_normalize(items)

    # renomeia colunas com "."
    df = df.rename(columns={
        "external_urls.spotify": "spotify_url"
    })

    # Pega a imagem de maior resolução
    df["image_url"] = df["images"].apply(
        lambda imgs: imgs[0]["url"] if isinstance(imgs, list) and imgs else None
    )

    # seleciona as colunas
    df = df[[
        "id", "name", "type", "uri", "spotify_url", "image_url"
    ]]

    # removendo duplicatas
    df = df.drop_duplicates(subset="id")

    return df

função de tratamento - músicas mais ouvidas

In [3]:
def transform_tracks(data) -> pd.DataFrame:
    """
    Transforma os dados brutos de tracks da Bronze.
    Achata artista e álbum aninhados, converte duração e padroniza tipos.
    """
    items = data["items"] if isinstance(data, dict) and "items" in data else data

    # achata tudo, criando colunas, pois, o JSON da API do Spotify tem campos dentro de campos
    df = pd.json_normalize(items)

    # renomeia colunas com "."
    df = df.rename(columns={
        "external_urls.spotify":        "spotify_url",
        "external_ids.isrc":            "isrc",
        "album.id":                     "album_id",
        "album.name":                   "album_name",
        "album.album_type":             "album_type",
        "album.release_date":           "album_release_date",
        "album.total_tracks":           "album_total_tracks",
        "album.external_urls.spotify":  "album_spotify_url",
    })

    # Artista principal, pega o id do primeiro artista
    df["artist_id"]   = df["artists"].apply(lambda a: a[0]["id"]   if isinstance(a, list) and a else None)
    df["artist_name"] = df["artists"].apply(lambda a: a[0]["name"] if isinstance(a, list) and a else None)

    # Imagem do álbum
    df["album_image_url"] = df["album.images"].apply(
        lambda imgs: imgs[0]["url"] if isinstance(imgs, list) and imgs else None
    )

    # Conversão de duração
    df["duration_ms"]  = df["duration_ms"].astype(int)
    df["duration_sec"] = (df["duration_ms"] / 1000).round(0).astype(int)
    df["duration_min"] = (df["duration_ms"] / 60000).round(2)

    #selecionando as apenas colunas úteis
    df = df[[
        "id", "name",
        "artist_id", "artist_name",
        "album_id", "album_name", "album_type", "album_release_date", "album_total_tracks", "album_image_url",
        "duration_ms", "duration_sec", "duration_min",
        "explicit", "is_local",
        "isrc", "spotify_url", "uri"
    ]]

    # Removendo duplicatas
    df = df.drop_duplicates(subset="id")
    df["explicit"]   = df["explicit"].astype(bool)
    df["is_local"]   = df["is_local"].astype(bool)

    return df

função de tratamento - últimas músicas ouvidas

In [4]:
def transform_recently_played(data) -> pd.DataFrame:
    """
    Transforma o histórico de reprodução da Bronze.
    Extrai dados do track aninhado, converte played_at e cria campos temporais.
    """
    items = data["items"] if isinstance(data, dict) and "items" in data else data

    # achata tudo, criando colunas, pois, o JSON da API do Spotify tem campos dentro de campos
    df = pd.json_normalize(items)

    # renomeia colunas com "."
    df = df.rename(columns={
        "track.id":           "track_id",
        "track.name":         "track_name",
        "track.duration_ms":  "duration_ms",
        "track.explicit":     "explicit",
        "track.is_local":     "is_local",
        "track.popularity":   "popularity",
        "track.uri":          "track_uri",
        "track.album.id":     "album_id",
        "track.album.name":   "album_name",
        "track.album.album_type":     "album_type",
        "track.album.release_date":   "album_release_date",
        "context.type":       "context_type",
        "context.uri":        "context_uri",
    })

    # Artista principal do track
    df["artist_id"]   = df["track.artists"].apply(lambda a: a[0]["id"]   if isinstance(a, list) and a else None)
    df["artist_name"] = df["track.artists"].apply(lambda a: a[0]["name"] if isinstance(a, list) and a else None)

    # Conversão de duração
    df["duration_ms"]  = df["duration_ms"].astype(int)
    df["duration_sec"] = (df["duration_ms"] / 1000).round(0).astype(int)
    df["duration_min"] = (df["duration_ms"] / 60000).round(2)

    # Converte played_at para datetime e extrai componentes temporais
    df["played_at"]      = pd.to_datetime(df["played_at"], utc=True)
    df["played_date"]    = df["played_at"].dt.date
    df["played_hour"]    = df["played_at"].dt.hour
    df["played_weekday"] = df["played_at"].dt.day_name()
    df["played_month"]   = df["played_at"].dt.month

    df = df[[
        "track_id", "track_name",
        "artist_id", "artist_name",
        "album_id", "album_name", "album_type", "album_release_date",
        "duration_ms", "duration_sec", "duration_min",
        "played_at", "played_date", "played_hour", "played_weekday", "played_month",
        "explicit", "is_local",
        "context_type", "context_uri", "track_uri"
    ]]

    # Chave única: mesma música não pode ter sido tocada no mesmo instante
    df = df.drop_duplicates(subset=["track_id", "played_at"])
    df["explicit"] = df["explicit"].astype(bool)
    df["is_local"] = df["is_local"].astype(bool)

    return df

Aplicando as funções e carregando no bucket

In [5]:
load_dotenv() # carrega .env

minio = MinioClient()
bucket = os.getenv("BUCKET_NAME")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# pegando os últimos arquivos json salvos de cada pasta da camada bronze
artists = minio.get_ultimo_arquivo(bucket, "bronze/artists")
tracks = minio.get_ultimo_arquivo(bucket, "bronze/tracks")
recently_played = minio.get_ultimo_arquivo(bucket, "bronze/recently_played")

# aplicando as funções de tratamento e transformando em dataframe
df_artists = transform_artists(artists)
df_tracks = transform_tracks(tracks)
df_recently = transform_recently_played(recently_played)

# Salva na Silver como Parquet
minio.upload_parquet(bucket, f"silver/artists/artists_{timestamp}.parquet", df_artists)
minio.upload_parquet(bucket, f"silver/tracks/tracks_{timestamp}.parquet", df_tracks)
minio.upload_parquet(bucket, f"silver/recently_played/recently_played_{timestamp}.parquet", df_recently)

[OK] bronze/artists/artists_20260424_151010.json | 2026-04-24 18:10:10
[OK] bronze/tracks/tracks_20260424_151010.json | 2026-04-24 18:10:10
[OK] bronze/recently_played/recently_played_20260424_151010.json | 2026-04-24 18:10:10
Parquet salvo em: spotify-data-pipeline/silver/artists/artists_20260424_175836.parquet
Parquet salvo em: spotify-data-pipeline/silver/tracks/tracks_20260424_175836.parquet
Parquet salvo em: spotify-data-pipeline/silver/recently_played/recently_played_20260424_175836.parquet


True